# 🌲 ROS 2 BehaviorTree.CPP 自定义节点完全指南

> **版本**: ROS 2 Humble/Iron + BehaviorTree.CPP v4  
> **作者**: 基于实战开发经验整理  
> **最后更新**: 2026年2月

---

## 📋 教程概述

本教程是一份**从零到实战**的 BehaviorTree.CPP 自定义节点编写指南。不仅提供代码模板，更深入剖析**为什么这样写**，帮助你理解背后的 C++ 机制和 ROS 2 设计哲学。

### 🎯 学习目标
完成本教程后，你将能够：
- ✅ 独立编写四种类型的行为树节点（Condition、Action、Control、Decorator）
- ✅ 理解 BehaviorTree.CPP 的核心概念（Blackboard、Port、NodeStatus）
- ✅ 掌握 CMake 构建系统的配置原理
- ✅ 熟练使用 ROS 2 与行为树的集成工具（`behaviortree_ros2`）
- ✅ 调试行为树节点的常见问题

### 📚 前置知识要求
- **C++ 基础**：继承、虚函数、命名空间、智能指针
- **ROS 2 基础**：熟悉 `rclcpp`、消息（msg）、CMake 构建
- **行为树概念**：了解 Sequence、Fallback 等标准控制节点（推荐先阅读 [BehaviorTree.CPP 官方文档](https://www.behaviortree.dev/)）

---

## 📖 教程大纲

| 章节 | 内容概要 | 难度 |
|------|---------|------|
| **1. 环境准备** | 项目结构、依赖安装 | ⭐ |
| **2. Condition Node** | 最简单的节点，布尔检查 | ⭐ |
| **3. Action Node** | 执行任务，支持异步操作 | ⭐⭐ |
| **4. Control Node** | 自定义控制流，管理子节点 | ⭐⭐⭐ |
| **5. Decorator Node** | 修饰子节点行为 | ⭐⭐ |
| **6. CMake 配置** | 编译系统深度解析 | ⭐⭐ |
| **7. 注册与调试** | 工厂模式、XML 配置、常见错误 | ⭐⭐⭐ |
| **8. 高级技巧** | ROS 2 集成、性能优化 | ⭐⭐⭐⭐ |

---

## 1. 🛠️ 预备知识与环境准备

### 1.1 核心概念速览

在开始编写代码前，先理解几个关键概念：

#### 📦 **Blackboard（黑板）**
- **作用**：全局共享的数据存储，类似于 ROS 的 Parameter Server。
- **使用场景**：不同节点之间传递数据（如导航目标、传感器数据）。
- **C++ 类型**：`BT::Blackboard`，通过 Port 机制访问。

#### 🔌 **Port（端口）**
- **InputPort**：从 Blackboard 读取数据（只读）。
- **OutputPort**：向 Blackboard 写入数据。
- **BidirectionalPort**：读写双向端口（较少用）。
- **为什么需要？** 提供类型安全的数据访问，避免手动 `static_cast`。

#### 📊 **NodeStatus（节点状态）**
```cpp
enum class NodeStatus {
    SUCCESS,   // ✅ 任务成功完成
    FAILURE,   // ❌ 任务失败
    RUNNING,   // ⏳ 任务进行中（异步）
    IDLE       // 😴 未激活（内部状态）
};
```

### 1.2 ROS 2 与 BehaviorTree 的集成

在 ROS 2 中，我们使用 `behaviortree_ros2` 包，它提供：
- **`RosTopicSubNode`**：自动订阅 ROS 话题并更新 Blackboard。
- **`RosTopicPubNode`**：从 Blackboard 读取数据并发布。
- **`RosServiceNode`**：调用 ROS 服务。
- **`RosActionNode`**：与 Action Server 交互（如 `nav2` 导航）。

> **💡 设计哲学**：行为树负责决策逻辑，ROS 负责硬件交互。两者通过 Blackboard 解耦。

### 1.3 推荐的项目结构

```text
src/rm_behavior_tree/rm_behavior_tree/
├── include/rm_behavior_tree/       # 公共头文件（对外接口）
│   └── plugins/
│       ├── action/
│       │   └── send_goal.hpp       # Action 节点声明
│       ├── condition/
│       │   └── is_control_zone_detected.hpp
│       ├── control/
│       │   └── decision_switch.hpp
│       └── decorator/
│           └── rate_controller.hpp
│
├── plugins/                         # 实现文件（内部逻辑）
│   ├── action/
│   │   └── send_goal.cpp           # 实现细节
│   ├── condition/
│   │   └── is_control_zone_detected.cpp
│   ├── control/
│   │   └── decision_switch.cpp
│   └── decorator/
│       └── rate_controller.cpp
│
├── src/
│   └── rm_behavior_tree.cpp        # 主程序：注册所有节点
│
├── config/
│   ├── 3v3_new.xml                 # 行为树配置文件
│   └── params.yaml                 # ROS 参数
│
├── CMakeLists.txt                  # 构建配置
└── package.xml                     # ROS 包描述
```

#### 🤔 **为什么要分离 `.hpp` 和 `.cpp`？**

| 方面 | 全头文件实现 | 分离声明与实现 |
|------|-------------|--------------|
| **编译速度** | ❌ 修改实现需重新编译所有依赖 | ✅ 只重新编译 `.cpp` |
| **二进制大小** | ❌ 每个编译单元都包含实现代码 | ✅ 代码只编译一次 |
| **接口清晰度** | ❌ 实现细节暴露 | ✅ `.hpp` 作为文档 |
| **模板支持** | ✅ 模板必须在头文件 | ⚠️ 显式实例化可分离 |

> **最佳实践**：对于模板类（如泛型算法），可以在 `.hpp` 实现；对于具体业务节点，**强烈推荐分离**。

### 1.4 依赖安装检查

运行以下命令确保环境就绪：

```bash
# 检查 BehaviorTree.CPP 版本
ros2 pkg list | grep behaviortree

# 安装依赖（如果缺失）
sudo apt install ros-humble-behaviortree-cpp-v3 ros-humble-behaviortree-ros2

# 验证编译环境
colcon build --packages-select rm_behavior_tree --cmake-args -DCMAKE_EXPORT_COMPILE_COMMANDS=ON
```

## 2. ✅ Condition Node (条件节点) - 最简单的开始

### 2.1 设计哲学

**条件节点的本质**：快速、无副作用的布尔检查。

#### ✅ 适用场景
- 检查传感器数据（"是否检测到敌人？"）
- 判断任务状态（"是否到达目标点？"）
- 验证前置条件（"电池电量是否充足？"）

#### ❌ 不应该做的事
- 发布 ROS 消息（应该用 Action）
- 执行耗时计算（超过 1ms）
- 修改机器人状态（应该用 Action）
- 返回 `RUNNING`（Condition 必须瞬间返回）

### 2.2 实战案例：`IsControlZoneDetected`

**需求**：在 RoboMaster 比赛中，检查机器人是否占领了指定区域。

#### 📄 头文件模板 (`include/rm_behavior_tree/plugins/condition/is_control_zone_detected.hpp`)

```cpp
#ifndef IS_CONTROL_ZONE_DETECTED_HPP_
#define IS_CONTROL_ZONE_DETECTED_HPP_

// 核心行为树库
#include <behaviortree_cpp_v3/condition_node.h>
// ROS 2 核心库
#include <rclcpp/rclcpp.hpp>
// 自定义消息接口
#include "rm_decision_interfaces/msg/rmul.hpp" 

namespace rm_behavior_tree  // 防止命名冲突
{
    /**
     * @brief 检查控制区是否被占领
     * @details 从 Blackboard 读取游戏状态信息，检查 control_zone_captured 字段
     */
    class IsControlZoneDetected : public BT::ConditionNode
    {
    public:
        /**
         * @brief 构造函数
         * @param name 节点在行为树中的名称（来自 XML）
         * @param config 包含端口配置和黑板引用
         */
        IsControlZoneDetected(const std::string& name, 
                              const BT::NodeConfiguration& config);

        /**
         * @brief 核心执行逻辑
         * @return SUCCESS 如果占领区被检测到，否则 FAILURE
         */
        BT::NodeStatus tick() override;

        /**
         * @brief 定义节点的输入/输出端口
         * @return 端口列表（静态方法，在节点实例化前调用）
         */
        static BT::PortsList providedPorts()
        {
            return {
                BT::InputPort<rm_decision_interfaces::msg::RMUL>(
                    "game_info",  // 端口名称（XML 中引用）
                    "Game state information from referee system"  // 描述（可选）
                )
            };
        }
    };
}

#endif  // IS_CONTROL_ZONE_DETECTED_HPP_
```

---

### 2.3 关键 C++ 语法深度解析

#### 🔍 **1. `override` 关键字的重要性**

```cpp
BT::NodeStatus tick() override;  // ✅ 推荐
BT::NodeStatus tick();           // ⚠️ 编译通过但危险
```

**`override` 的作用**：
- **编译时检查**：如果基类没有这个虚函数（例如拼写错误 `tik()`），编译器会报错。
- **防止隐藏**：如果基类签名改变（如添加 `const`），编译器会提醒你。

**反面案例**：
```cpp
// 基类
class Base { virtual void process(int x); };

// 错误：忘记参数，但编译通过（隐藏而非重写）
class Derived : public Base {
    void process() { /* ... */ }  // 这不是重写！
};

// 正确：编译器会报错
class Derived : public Base {
    void process() override { /* ... */ }  // ❌ 编译失败：签名不匹配
};
```

---

#### 🔍 **2. `static` 方法与工厂模式**

```cpp
static BT::PortsList providedPorts() { /* ... */ }
```

**为什么必须是 `static`？**

行为树在运行前需要验证 XML 配置的正确性：

```cpp
// BehaviorTreeFactory 内部逻辑（简化版）
void BehaviorTreeFactory::registerNodeType<T>(const std::string& id) {
    // 注意：这里还没有创建节点实例！
    PortsList ports = T::providedPorts();  // 必须是静态方法
    
    // 检查 XML 中使用的端口是否都在 ports 中定义
    validateXMLAgainstPorts(id, ports);
}
```

如果 `providedPorts()` 不是静态的，就需要先 `new T()`，但这时还没有 `NodeConfiguration`，无法构造对象 → **死锁**。

---

#### 🔍 **3. 端口类型系统**

```cpp
BT::InputPort<rm_decision_interfaces::msg::RMUL>("game_info")
```

**端口的类型安全机制**：

```cpp
// 内部实现（简化）
template<typename T>
class InputPort {
    std::type_index type_info = typeid(T);  // 存储类型信息
    
    bool validateType(const std::any& value) {
        return value.type() == type_info;  // 运行时检查
    }
};
```

**实战示例**：类型不匹配会怎样？

```xml
<!-- XML 配置 -->
<IsControlZoneDetected game_info="{wrong_data}"/>
```

```cpp
// Blackboard 中存储的是 std::string
blackboard->set("wrong_data", std::string("hello"));

// C++ 代码尝试读取 RMUL 类型
rm_decision_interfaces::msg::RMUL game_info;
if (!getInput("game_info", game_info)) {
    // ⚠️ getInput 返回 false，因为类型不匹配
    RCLCPP_ERROR(logger_, "Type mismatch for port 'game_info'");
}
```

---

### 2.4 源文件实现 (`plugins/condition/is_control_zone_detected.cpp`)

```cpp
#include "rm_behavior_tree/plugins/condition/is_control_zone_detected.hpp"

namespace rm_behavior_tree
{

IsControlZoneDetected::IsControlZoneDetected(
    const std::string& name, 
    const BT::NodeConfiguration& config)
    : BT::ConditionNode(name, config)  // ⚠️ 必须初始化基类
{
    // 可选：在构造函数中做初始化工作
    // 例如：创建 ROS 订阅器、分配内存等
    // 但注意：构造函数不应做耗时操作！
}

BT::NodeStatus IsControlZoneDetected::tick()
{
    // === 第 1 步：从 Blackboard 获取数据 ===
    rm_decision_interfaces::msg::RMUL game_info;
    
    // getInput 的内部逻辑：
    // 1. 查找 XML 中 game_info="{xxx}" 指向的黑板键
    // 2. 从黑板获取对应的值
    // 3. 尝试转换为 RMUL 类型
    if (!getInput("game_info", game_info))
    {
        // 这是严重的配置错误！可能原因：
        // 1. XML 中未绑定黑板键：<IsControlZoneDetected game_info="{key}"/>
        // 2. 黑板上没有这个键
        // 3. 类型不匹配
        
        // 调试技巧：打印详细错误
        RCLCPP_ERROR(
            rclcpp::get_logger("IsControlZoneDetected"),
            "Failed to get input 'game_info'. Check XML configuration!"
        );
        
        return BT::NodeStatus::FAILURE;
    }

    // === 第 2 步：业务逻辑检查 ===
    if (game_info.control_zone_captured) 
    {
        // 占领区被占领，条件满足
        return BT::NodeStatus::SUCCESS;
    }
    else
    {
        // 占领区未被占领，条件不满足
        return BT::NodeStatus::FAILURE;
    }
    
    // ⚠️ Condition 节点禁止返回 RUNNING！
    // return BT::NodeStatus::RUNNING;  // ❌ 编译通过但语义错误
}

}  // namespace rm_behavior_tree
```

---

### 2.5 常见错误与调试技巧

#### ❌ **错误 1：忘记在 XML 中绑定黑板键**

```xml
<!-- 错误：直接传递字面值 -->
<IsControlZoneDetected game_info="some_string"/>

<!-- 正确：绑定到黑板 -->
<IsControlZoneDetected game_info="{game_state}"/>
```

#### ❌ **错误 2：在 Condition 中执行耗时操作**

```cpp
// ❌ 错误示例
BT::NodeStatus tick() {
    // 这会阻塞整个行为树！
    std::this_thread::sleep_for(std::chrono::seconds(5));
    return BT::NodeStatus::SUCCESS;
}
```

**正确做法**：如果需要等待，使用 `BT::StatefulActionNode` 或 `Decorator`（如 `Timeout`）。

#### 🐛 **调试技巧**

```cpp
// 添加详细日志
BT::NodeStatus tick() {
    RCLCPP_DEBUG(logger_, "[%s] Checking control zone...", name().c_str());
    
    rm_decision_interfaces::msg::RMUL game_info;
    if (!getInput("game_info", game_info)) {
        RCLCPP_WARN(logger_, "Input 'game_info' not available");
        return BT::NodeStatus::FAILURE;
    }
    
    RCLCPP_DEBUG(logger_, "Control zone status: %s", 
                 game_info.control_zone_captured ? "CAPTURED" : "NOT CAPTURED");
    
    return game_info.control_zone_captured ? 
           BT::NodeStatus::SUCCESS : BT::NodeStatus::FAILURE;
}
```

## 3. ⚡ Action Node (动作节点) - 行为树的执行者

### 3.1 Action vs Condition：关键区别

| 特性 | Condition Node | Action Node |
|------|---------------|-------------|
| **语义** | 检查状态（"是什么"） | 执行任务（"做什么"） |
| **返回 RUNNING** | ❌ 禁止 | ✅ 允许（异步） |
| **副作用** | ❌ 无副作用 | ✅ 修改世界状态 |
| **执行时间** | < 1ms（瞬时） | 可能数秒甚至数分钟 |
| **典型用途** | `if (battery > 20%)` | `navigate_to(point)` |

### 3.2 两种 Action 类型

#### 📌 **类型 1：SyncActionNode（同步动作）**
- 立即完成，类似于 Condition。
- 适用于：设置变量、简单计算、发布单次消息。

```cpp
class SetBlackboardValue : public BT::SyncActionNode {
    BT::NodeStatus tick() override {
        setOutput("result", 42);  // 瞬间完成
        return BT::NodeStatus::SUCCESS;
    }
};
```

#### 📌 **类型 2：StatefulActionNode（状态机动作）**
- 支持异步操作，可返回 `RUNNING`。
- 适用于：导航、抓取、等待传感器数据。

**状态转换图**：
```
  [IDLE]
    ↓ (首次 tick)
  onStart() → RUNNING/SUCCESS/FAILURE
    ↓ (后续 tick，如果返回 RUNNING)
  onRunning() → RUNNING/SUCCESS/FAILURE
    ↓ (被中断时)
  onHalted() → IDLE
```

---

### 3.3 实战案例 1：同步动作（`ClearRecoveryFlag`）

**需求**：重置黑板上的恢复标志位。

```cpp
// include/rm_behavior_tree/plugins/action/clear_recovery_flag.hpp
#ifndef CLEAR_RECOVERY_FLAG_HPP_
#define CLEAR_RECOVERY_FLAG_HPP_

#include <behaviortree_cpp_v3/action_node.h>

namespace rm_behavior_tree {

class ClearRecoveryFlag : public BT::SyncActionNode
{
public:
    ClearRecoveryFlag(const std::string& name, const BT::NodeConfiguration& config)
        : BT::SyncActionNode(name, config) {}

    static BT::PortsList providedPorts() {
        return {
            BT::InputPort<std::string>("flag_name", "recovery_needed", "黑板键名")
        };
    }

    BT::NodeStatus tick() override;
};

}  // namespace rm_behavior_tree
#endif
```

```cpp
// plugins/action/clear_recovery_flag.cpp
#include "rm_behavior_tree/plugins/action/clear_recovery_flag.hpp"

BT::NodeStatus ClearRecoveryFlag::tick()
{
    std::string flag_name;
    if (!getInput("flag_name", flag_name)) {
        flag_name = "recovery_needed";  // 使用默认值
    }

    // 直接操作黑板（高级用法）
    config().blackboard->set(flag_name, false);
    
    RCLCPP_INFO(rclcpp::get_logger("ClearRecoveryFlag"), 
                "Cleared flag: %s", flag_name.c_str());
    
    return BT::NodeStatus::SUCCESS;  // 瞬间完成
}
```

---

### 3.4 实战案例 2：异步动作（`SendGoalToNav2`）

**需求**：向 Nav2 导航栈发送目标点，等待到达。

#### 📄 头文件 (`include/.../send_goal_to_nav2.hpp`)

```cpp
#ifndef SEND_GOAL_TO_NAV2_HPP_
#define SEND_GOAL_TO_NAV2_HPP_

#include <behaviortree_cpp_v3/action_node.h>
#include <rclcpp/rclcpp.hpp>
#include <geometry_msgs/msg/pose_stamped.hpp>
#include <rclcpp_action/rclcpp_action.hpp>
#include <nav2_msgs/action/navigate_to_pose.hpp>

namespace rm_behavior_tree {

class SendGoalToNav2 : public BT::StatefulActionNode
{
public:
    using NavigateToPose = nav2_msgs::action::NavigateToPose;
    using GoalHandle = rclcpp_action::ClientGoalHandle<NavigateToPose>;

    SendGoalToNav2(const std::string& name, 
                   const BT::NodeConfiguration& config,
                   rclcpp::Node::SharedPtr node);  // ⚠️ 需要 ROS 节点

    static BT::PortsList providedPorts() {
        return {
            BT::InputPort<geometry_msgs::msg::PoseStamped>("goal_pose"),
            BT::InputPort<double>("timeout", 30.0, "超时时间（秒）"),
            BT::OutputPort<std::string>("failure_reason", "失败原因")
        };
    }

    // 状态机三大核心方法
    BT::NodeStatus onStart() override;
    BT::NodeStatus onRunning() override;
    void onHalted() override;

private:
    rclcpp::Node::SharedPtr node_;
    rclcpp_action::Client<NavigateToPose>::SharedPtr action_client_;
    std::shared_ptr<GoalHandle> goal_handle_;
    std::chrono::steady_clock::time_point start_time_;
    double timeout_;
};

}  // namespace rm_behavior_tree
#endif
```

---

#### 📄 实现文件 (`plugins/action/send_goal_to_nav2.cpp`)

```cpp
#include "rm_behavior_tree/plugins/action/send_goal_to_nav2.hpp"

namespace rm_behavior_tree {

SendGoalToNav2::SendGoalToNav2(
    const std::string& name, 
    const BT::NodeConfiguration& config,
    rclcpp::Node::SharedPtr node)
    : BT::StatefulActionNode(name, config), node_(node)
{
    // 在构造函数中创建 Action Client
    action_client_ = rclcpp_action::create_client<NavigateToPose>(
        node_, "navigate_to_pose");
    
    // ⚠️ 注意：不要在这里阻塞等待服务器！
    // 应该在 onStart() 中检查
}

// ========== 状态机方法 1：启动阶段 ==========
BT::NodeStatus SendGoalToNav2::onStart()
{
    RCLCPP_INFO(node_->get_logger(), "[%s] Starting navigation...", name().c_str());
    
    // === 步骤 1：获取输入参数 ===
    geometry_msgs::msg::PoseStamped goal_pose;
    if (!getInput("goal_pose", goal_pose)) {
        setOutput("failure_reason", "Missing goal_pose input");
        RCLCPP_ERROR(node_->get_logger(), "No goal pose provided!");
        return BT::NodeStatus::FAILURE;
    }
    
    if (!getInput("timeout", timeout_)) {
        timeout_ = 30.0;  // 默认 30 秒
    }
    
    // === 步骤 2：检查 Action Server 是否可用 ===
    if (!action_client_->wait_for_action_server(std::chrono::seconds(1))) {
        setOutput("failure_reason", "Nav2 action server not available");
        RCLCPP_ERROR(node_->get_logger(), "Nav2 is not running!");
        return BT::NodeStatus::FAILURE;
    }
    
    // === 步骤 3：发送目标 ===
    auto goal_msg = NavigateToPose::Goal();
    goal_msg.pose = goal_pose;
    
    auto send_goal_options = rclcpp_action::Client<NavigateToPose>::SendGoalOptions();
    
    // 设置回调函数（可选，用于调试）
    send_goal_options.feedback_callback = 
        [this](auto, const auto& feedback) {
            RCLCPP_DEBUG(node_->get_logger(), 
                        "Distance remaining: %.2f m", 
                        feedback->distance_remaining);
        };
    
    // 异步发送（不阻塞）
    auto goal_handle_future = action_client_->async_send_goal(goal_msg, send_goal_options);
    
    // ⚠️ 这里不能 wait()，否则会阻塞行为树！
    // 应该在 onRunning() 中检查结果
    
    // 等待 goal 被接受（快速操作，通常 < 10ms）
    if (rclcpp::spin_until_future_complete(node_, goal_handle_future, std::chrono::milliseconds(100)) 
        != rclcpp::FutureReturnCode::SUCCESS) {
        setOutput("failure_reason", "Failed to send goal");
        return BT::NodeStatus::FAILURE;
    }
    
    goal_handle_ = goal_handle_future.get();
    if (!goal_handle_) {
        setOutput("failure_reason", "Goal rejected by server");
        return BT::NodeStatus::FAILURE;
    }
    
    // 记录开始时间（用于超时检测）
    start_time_ = std::chrono::steady_clock::now();
    
    // 告诉行为树：任务已启动，但还没完成
    return BT::NodeStatus::RUNNING;
}

// ========== 状态机方法 2：运行阶段 ==========
BT::NodeStatus SendGoalToNav2::onRunning()
{
    // 每次 tick 都会调用这个方法，直到返回 SUCCESS 或 FAILURE
    
    // === 检查 1：超时检测 ===
    auto elapsed = std::chrono::steady_clock::now() - start_time_;
    if (std::chrono::duration<double>(elapsed).count() > timeout_) {
        RCLCPP_WARN(node_->get_logger(), "[%s] Navigation timeout!", name().c_str());
        action_client_->async_cancel_goal(goal_handle_);  // 取消目标
        setOutput("failure_reason", "Timeout");
        return BT::NodeStatus::FAILURE;
    }
    
    // === 检查 2：任务是否完成 ===
    auto result_future = action_client_->async_get_result(goal_handle_);
    
    // 非阻塞检查（立即返回）
    if (result_future.wait_for(std::chrono::milliseconds(0)) == std::future_status::ready) {
        auto result = result_future.get();
        
        switch (result.code) {
            case rclcpp_action::ResultCode::SUCCEEDED:
                RCLCPP_INFO(node_->get_logger(), "[%s] Navigation succeeded!", name().c_str());
                return BT::NodeStatus::SUCCESS;
                
            case rclcpp_action::ResultCode::ABORTED:
                setOutput("failure_reason", "Aborted by Nav2");
                return BT::NodeStatus::FAILURE;
                
            case rclcpp_action::ResultCode::CANCELED:
                setOutput("failure_reason", "Canceled");
                return BT::NodeStatus::FAILURE;
                
            default:
                setOutput("failure_reason", "Unknown error");
                return BT::NodeStatus::FAILURE;
        }
    }
    
    // 任务还在进行中
    return BT::NodeStatus::RUNNING;
}

// ========== 状态机方法 3：中断处理 ==========
void SendGoalToNav2::onHalted()
{
    // ⚠️ 非常重要！必须实现这个方法！
    // 当父节点（如 Sequence）决定切换到另一个分支时，会调用这里
    
    RCLCPP_WARN(node_->get_logger(), "[%s] Navigation halted! Canceling goal...", name().c_str());
    
    if (goal_handle_) {
        // 向 Nav2 发送取消请求
        auto cancel_future = action_client_->async_cancel_goal(goal_handle_);
        
        // 等待取消完成（最多 500ms）
        if (rclcpp::spin_until_future_complete(node_, cancel_future, std::chrono::milliseconds(500))
            == rclcpp::FutureReturnCode::SUCCESS) {
            RCLCPP_INFO(node_->get_logger(), "Goal canceled successfully");
        } else {
            RCLCPP_ERROR(node_->get_logger(), "Failed to cancel goal!");
        }
        
        goal_handle_.reset();
    }
    
    // ⚠️ 如果不实现 onHalted，机器人可能会：
    // 1. 继续导航到旧目标（即使行为树已切换任务）
    // 2. 资源泄漏（goal_handle_ 未释放）
}

}  // namespace rm_behavior_tree
```

---

### 3.5 关键设计模式解析

#### 🔍 **为什么需要 onStart/onRunning 分离？**

**反面案例**（单一 `tick()` 方法）：
```cpp
// ❌ 糟糕的设计
BT::NodeStatus tick() {
    if (!goal_sent_) {
        send_goal();
        goal_sent_ = true;
    }
    
    if (is_goal_reached()) {
        goal_sent_ = false;  // 重置状态
        return SUCCESS;
    }
    return RUNNING;
}
```

**问题**：
1. 需要手动维护 `goal_sent_` 标志（容易出错）。
2. 状态重置逻辑分散（`goal_sent_ = false`）。
3. 无法优雅处理中断（缺少清理逻辑）。

**正确设计**（状态机模式）：
- `onStart`：自动在首次激活时调用，无需手动标志位。
- `onRunning`：每次 tick 调用，逻辑清晰。
- `onHalted`：自动在中断时调用，保证清理。

---

### 3.6 常见陷阱与最佳实践

#### ❌ **陷阱 1：在 onStart 中阻塞**
```cpp
// ❌ 错误
BT::NodeStatus onStart() {
    auto future = action_client_->async_send_goal(goal);
    future.wait();  // 阻塞！行为树无法 tick 其他节点
    return RUNNING;
}

// ✅ 正确
BT::NodeStatus onStart() {
    action_client_->async_send_goal(goal);  // 立即返回
    return RUNNING;  // 在 onRunning 中检查结果
}
```

#### ❌ **陷阱 2：忘记实现 onHalted**
```cpp
// 如果不实现 onHalted，当行为树中断时：
// - ROS Action 继续执行（机器人失控）
// - 资源未释放（内存泄漏）

// ✅ 始终实现清理逻辑
void onHalted() override {
    cancel_all_pending_operations();
    release_resources();
}
```

#### ✅ **最佳实践：超时保护**
```cpp
BT::NodeStatus onRunning() {
    if (elapsed_time() > timeout_) {
        cleanup();
        return FAILURE;
    }
    // ...
}
```

## 4. 🎛️ Control Node (控制节点) - 编排决策流程

### 4.1 控制节点的本质

**作用**：管理多个子节点的执行顺序和逻辑。

#### 标准控制节点回顾
| 节点类型 | 逻辑 | 应用场景 |
|---------|------|---------|
| **Sequence** | 依次执行，遇到 FAILURE 中断 | 严格的步骤流程 |
| **Fallback** | 依次尝试，遇到 SUCCESS 中断 | 备选方案 |
| **Parallel** | 同时执行所有子节点 | 并发任务 |
| **ReactiveFallback** | 每次 tick 重新评估所有子节点 | 动态优先级 |

### 4.2 何时需要自定义控制节点？

❌ **不推荐自定义的情况**（99% 的需求都能用标准节点实现）：
- 简单的顺序逻辑 → 用 `Sequence`
- 条件分支 → 用 `Fallback` + `Condition`
- 循环 → 用 `Repeat` 或 `RetryUntilSuccessful`

✅ **推荐自定义的情况**：
- 需要**动态选择子节点**（如基于优先级的策略切换）
- 需要**特殊的中断逻辑**（如"完成一半后不可中断"）
- 需要**状态持久化**（如"记住上次执行到哪个子节点"）

---

### 4.3 实战案例：`DecisionSwitch`（动态策略切换）

**需求**：根据黑板上的 `strategy` 字段，执行不同的子树。

```
DecisionSwitch
  ├─ AttackStrategy   (strategy == "attack")
  ├─ DefenseStrategy  (strategy == "defense")
  └─ IdleStrategy     (strategy == "idle")
```

#### 📄 头文件 (`include/.../decision_switch.hpp`)

```cpp
#ifndef DECISION_SWITCH_HPP_
#define DECISION_SWITCH_HPP_

#include <behaviortree_cpp_v3/control_node.h>
#include <rclcpp/rclcpp.hpp>

namespace rm_behavior_tree {

/**
 * @brief 基于黑板值动态选择子节点
 * @details 类似于 switch-case 语句，但在行为树中实现
 */
class DecisionSwitch : public BT::ControlNode
{
public:
    DecisionSwitch(const std::string& name, const BT::NodeConfiguration& config)
        : BT::ControlNode(name, config), current_child_idx_(-1)
    {}

    static BT::PortsList providedPorts() {
        return {
            BT::InputPort<std::string>("strategy", "当前策略名称"),
            BT::InputPort<std::vector<std::string>>("child_names", "子节点名称列表")
        };
    }

    // 控制节点核心方法
    BT::NodeStatus tick() override;
    
    // 中断处理
    void halt() override;

private:
    int current_child_idx_;  // 当前执行的子节点索引
};

}  // namespace rm_behavior_tree
#endif
```

---

#### 📄 实现文件 (`plugins/control/decision_switch.cpp`)

```cpp
#include "rm_behavior_tree/plugins/control/decision_switch.hpp"

namespace rm_behavior_tree {

BT::NodeStatus DecisionSwitch::tick()
{
    // === 步骤 1：获取当前策略 ===
    std::string current_strategy;
    if (!getInput("strategy", current_strategy)) {
        RCLCPP_ERROR(rclcpp::get_logger("DecisionSwitch"), 
                     "Missing required input 'strategy'");
        return BT::NodeStatus::FAILURE;
    }

    // === 步骤 2：获取子节点名称映射 ===
    std::vector<std::string> child_names;
    if (!getInput("child_names", child_names)) {
        // 如果没有指定，使用子节点索引
        child_names.resize(children_nodes_.size());
        for (size_t i = 0; i < children_nodes_.size(); ++i) {
            child_names[i] = children_nodes_[i]->name();
        }
    }

    // === 步骤 3：查找匹配的子节点 ===
    int target_idx = -1;
    for (size_t i = 0; i < child_names.size(); ++i) {
        if (child_names[i] == current_strategy) {
            target_idx = static_cast<int>(i);
            break;
        }
    }

    if (target_idx < 0 || target_idx >= static_cast<int>(children_nodes_.size())) {
        RCLCPP_WARN(rclcpp::get_logger("DecisionSwitch"),
                    "Strategy '%s' not found in children", current_strategy.c_str());
        return BT::NodeStatus::FAILURE;
    }

    // === 步骤 4：如果策略改变，中断旧节点 ===
    if (current_child_idx_ != target_idx && current_child_idx_ >= 0) {
        RCLCPP_INFO(rclcpp::get_logger("DecisionSwitch"),
                    "Strategy changed from %s to %s, halting old child",
                    child_names[current_child_idx_].c_str(),
                    child_names[target_idx].c_str());
        
        // ⚠️ 重要：必须显式中断旧节点
        haltChild(current_child_idx_);
    }

    current_child_idx_ = target_idx;

    // === 步骤 5：执行选中的子节点 ===
    TreeNode* child_node = children_nodes_[current_child_idx_];
    BT::NodeStatus child_status = child_node->executeTick();

    // 根据子节点状态决定返回值
    switch (child_status) {
        case BT::NodeStatus::RUNNING:
            return BT::NodeStatus::RUNNING;  // 子节点还在运行
            
        case BT::NodeStatus::SUCCESS:
            current_child_idx_ = -1;  // 重置状态
            return BT::NodeStatus::SUCCESS;
            
        case BT::NodeStatus::FAILURE:
            current_child_idx_ = -1;
            return BT::NodeStatus::FAILURE;
            
        default:
            return BT::NodeStatus::FAILURE;
    }
}

void DecisionSwitch::halt()
{
    // 当整个 DecisionSwitch 被中断时，必须中断当前子节点
    if (current_child_idx_ >= 0 && current_child_idx_ < static_cast<int>(children_nodes_.size())) {
        haltChild(current_child_idx_);
    }
    current_child_idx_ = -1;
    
    // ⚠️ 必须调用基类的 halt()
    BT::ControlNode::halt();
}

}  // namespace rm_behavior_tree
```

---

### 4.4 控制节点开发的关键点

#### 🔍 **1. 子节点的生命周期管理**

```cpp
// ✅ 正确：使用 executeTick()
BT::NodeStatus status = child_node->executeTick();

// ❌ 错误：直接调用 tick()（跳过状态管理）
BT::NodeStatus status = child_node->tick();  // 会导致状态不一致！
```

**`executeTick()` 做了什么？**
```cpp
// BehaviorTree.CPP 内部实现（简化版）
NodeStatus TreeNode::executeTick() {
    if (status_ == NodeStatus::IDLE) {
        status_ = NodeStatus::RUNNING;
        // 触发 onStart() 等生命周期回调
    }
    
    status_ = this->tick();  // 调用你的实现
    
    if (status_ != NodeStatus::RUNNING) {
        status_ = NodeStatus::IDLE;  // 重置状态
    }
    
    return status_;
}
```

#### 🔍 **2. 中断传播的正确姿势**

```cpp
void MyControlNode::halt() {
    // 步骤 1：中断所有活跃的子节点
    for (size_t i = 0; i < children_nodes_.size(); ++i) {
        if (children_nodes_[i]->status() == NodeStatus::RUNNING) {
            haltChild(i);  // ⚠️ 使用 haltChild()，而非直接调用 child->halt()
        }
    }
    
    // 步骤 2：重置自己的内部状态
    current_child_idx_ = -1;
    
    // 步骤 3：必须调用基类的 halt()
    ControlNode::halt();
}
```

---

## 5. 🎨 Decorator Node (装饰节点) - 增强子节点能力

### 5.1 装饰器模式的本质

**作用**：包装单个子节点，修改其行为或返回值。

#### 标准装饰器回顾
| 装饰器 | 功能 | 典型用途 |
|--------|------|---------|
| **Inverter** | 反转 SUCCESS ↔ FAILURE | `NOT` 逻辑 |
| **ForceSuccess** | 强制返回 SUCCESS | 忽略失败 |
| **Repeat** | 重复执行 N 次 | 循环任务 |
| **RetryUntilSuccessful** | 失败后重试 | 容错机制 |
| **Timeout** | 限制执行时间 | 防止死锁 |

### 5.2 实战案例：`RateController`（频率控制器）

**需求**：限制子节点的执行频率（例如：每秒最多执行 10 次）。

#### 📄 完整实现

```cpp
// include/rm_behavior_tree/plugins/decorator/rate_controller.hpp
#ifndef RATE_CONTROLLER_HPP_
#define RATE_CONTROLLER_HPP_

#include <behaviortree_cpp_v3/decorator_node.h>
#include <chrono>

namespace rm_behavior_tree {

class RateController : public BT::DecoratorNode
{
public:
    RateController(const std::string& name, const BT::NodeConfiguration& config)
        : BT::DecoratorNode(name, config), 
          last_execution_time_(std::chrono::steady_clock::time_point::min())
    {}

    static BT::PortsList providedPorts() {
        return {
            BT::InputPort<double>("hz", 1.0, "执行频率（Hz）")
        };
    }

    BT::NodeStatus tick() override {
        double hz;
        if (!getInput("hz", hz) || hz <= 0) {
            hz = 1.0;
        }

        auto now = std::chrono::steady_clock::now();
        auto interval = std::chrono::duration<double>(1.0 / hz);
        auto elapsed = now - last_execution_time_;

        if (elapsed < interval) {
            // 太频繁，跳过这次执行
            // ⚠️ 返回子节点的上一次状态（而非重新执行）
            return child_node_->status();
        }

        // 允许执行
        last_execution_time_ = now;
        return child_node_->executeTick();  // ⚠️ 必须用 executeTick()
    }

private:
    std::chrono::steady_clock::time_point last_execution_time_;
};

}  // namespace rm_behavior_tree
#endif
```

---

### 5.3 实战案例 2：`ConditionalDecorator`（条件装饰器）

**需求**：只有当某个条件满足时，才执行子节点。

```cpp
class ConditionalDecorator : public BT::DecoratorNode
{
public:
    // ...

    static BT::PortsList providedPorts() {
        return {
            BT::InputPort<bool>("condition", "是否执行子节点")
        };
    }

    BT::NodeStatus tick() override {
        bool condition;
        if (!getInput("condition", condition)) {
            return BT::NodeStatus::FAILURE;
        }

        if (!condition) {
            // 条件不满足，直接返回 FAILURE（不执行子节点）
            return BT::NodeStatus::FAILURE;
        }

        // 条件满足，执行子节点
        return child_node_->executeTick();
    }
};
```

**XML 使用示例**：
```xml
<ConditionalDecorator condition="{battery_ok}">
    <NavigateToGoal goal="{target}"/>
</ConditionalDecorator>
```

---

### 5.4 Decorator 开发要点

#### 🔍 **1. 子节点访问方式**

```cpp
// ✅ 正确
BT::NodeStatus tick() override {
    return child_node_->executeTick();  // child_node_ 是受保护成员
}

// ❌ 错误：DecoratorNode 没有 children_nodes_
for (auto* child : children_nodes_) { /* ... */ }  // 编译错误！
```

#### 🔍 **2. 状态缓存技巧**

```cpp
class StatefulDecorator : public BT::DecoratorNode {
    BT::NodeStatus tick() override {
        // 场景：希望子节点每 5 秒执行一次，其余时间返回缓存状态
        if (should_execute()) {
            cached_status_ = child_node_->executeTick();
        }
        return cached_status_;
    }

private:
    BT::NodeStatus cached_status_ = BT::NodeStatus::IDLE;
};
```

#### ⚠️ **常见错误：忘记传播 RUNNING**

```cpp
// ❌ 错误
BT::NodeStatus tick() override {
    auto status = child_node_->executeTick();
    if (status == BT::NodeStatus::SUCCESS) {
        return BT::NodeStatus::SUCCESS;
    }
    return BT::NodeStatus::FAILURE;  // 错误：RUNNING 被转换为 FAILURE！
}

// ✅ 正确
BT::NodeStatus tick() override {
    auto status = child_node_->executeTick();
    
    // 对于 RUNNING，始终透传
    if (status == BT::NodeStatus::RUNNING) {
        return BT::NodeStatus::RUNNING;
    }
    
    // 只修改终态（SUCCESS/FAILURE）
    return modify_status(status);
}
```

## 6. 🔧 CMakeLists.txt 配置详解 - 构建系统的奥秘

### 6.1 为什么要理解 CMake？

❌ **盲目复制粘贴的后果**：
- 编译错误："undefined reference to XXX"
- 链接错误："cannot find -lYYY"
- 运行时错误："symbol not found"

✅ **理解 CMake 后的收益**：
- 快速定位编译问题
- 优化编译速度
- 正确管理依赖关系

---

### 6.2 完整的 CMakeLists.txt 模板（逐行解析）

```cmake
# ========================================
# 1. 项目基本信息
# ========================================
cmake_minimum_required(VERSION 3.8)
project(rm_behavior_tree)

# C++ 标准设置（BehaviorTree.CPP 需要 C++17）
if(CMAKE_COMPILER_IS_GNUCXX OR CMAKE_CXX_COMPILER_ID MATCHES "Clang")
  add_compile_options(-Wall -Wextra -Wpedantic)  # 开启警告
endif()

set(CMAKE_CXX_STANDARD 17)  # ⚠️ 必须，否则无法编译 std::optional 等特性
set(CMAKE_CXX_STANDARD_REQUIRED ON)

# ========================================
# 2. 查找依赖包
# ========================================
# find_package 做了什么？
# 1. 在系统中搜索 <PackageName>Config.cmake 文件
# 2. 设置变量：${PackageName}_INCLUDE_DIRS, ${PackageName}_LIBRARIES
# 3. 如果 REQUIRED，找不到会报错并停止构建

find_package(ament_cmake REQUIRED)       # ROS 2 构建工具
find_package(rclcpp REQUIRED)            # ROS 2 C++ 客户端库
find_package(behaviortree_cpp_v3 REQUIRED)  # 行为树核心库

# 自定义消息接口（如果你用到）
find_package(rm_decision_interfaces REQUIRED)
find_package(geometry_msgs REQUIRED)     # 标准消息类型

# ROS 2 行为树集成库（提供 RosTopicSubNode 等）
find_package(behaviortree_ros2 REQUIRED)

# ========================================
# 3. 包含头文件目录
# ========================================
# 为什么需要这一行？
# 编译器在编译 .cpp 时，遇到 #include "xxx.hpp" 会在这些目录中查找

include_directories(
  include  # 我们的头文件（相对路径，相对于 CMakeLists.txt）
  # 注意：依赖包的头文件路径已经通过 find_package 自动添加了
)

# ========================================
# 4. 编译可执行文件或库
# ========================================
# 方式 1：创建可执行文件（用于节点）
add_executable(rm_behavior_tree_node
  src/rm_behavior_tree.cpp  # 主程序

  # ⚠️ 关键：必须列出所有 .cpp 文件！
  # 忘记添加会导致 "undefined reference" 错误
  plugins/condition/is_control_zone_detected.cpp
  plugins/condition/is_attacked.cpp
  
  plugins/action/send_goal.cpp
  plugins/action/clear_recovery_flag.cpp
  
  plugins/control/decision_switch.cpp
  
  plugins/decorator/rate_controller.cpp
)

# 方式 2：创建共享库（用于插件化架构）
# add_library(rm_behavior_tree_plugins SHARED
#   plugins/condition/is_control_zone_detected.cpp
#   # ...
# )

# ========================================
# 5. 链接依赖库
# ========================================
# 为什么需要这一步？
# 1. 编译时：需要知道函数签名（通过头文件）
# 2. 链接时：需要找到函数实现（通过库文件 .so/.a）

# ROS 2 推荐方式（自动处理 include 和 link）
ament_target_dependencies(rm_behavior_tree_node
  "rclcpp"
  "behaviortree_cpp_v3"
  "behaviortree_ros2"
  "rm_decision_interfaces"  # ⚠️ 如果用到自定义消息，必须链接！
  "geometry_msgs"
)

# 传统 CMake 方式（效果相同，但更繁琐）
# target_include_directories(rm_behavior_tree_node PUBLIC
#   ${rclcpp_INCLUDE_DIRS}
#   ${behaviortree_cpp_v3_INCLUDE_DIRS}
# )
# target_link_libraries(rm_behavior_tree_node
#   ${rclcpp_LIBRARIES}
#   ${behaviortree_cpp_v3_LIBRARIES}
# )

# ========================================
# 6. 安装规则
# ========================================
# 将编译好的文件安装到 ROS 2 工作空间

# 安装可执行文件
install(TARGETS rm_behavior_tree_node
  DESTINATION lib/${PROJECT_NAME}
)

# 安装启动文件
install(DIRECTORY launch
  DESTINATION share/${PROJECT_NAME}/
)

# 安装配置文件
install(DIRECTORY config
  DESTINATION share/${PROJECT_NAME}/
)

# 安装头文件（如果其他包需要依赖）
install(DIRECTORY include/
  DESTINATION include
)

# ========================================
# 7. 测试（可选）
# ========================================
if(BUILD_TESTING)
  find_package(ament_lint_auto REQUIRED)
  ament_lint_auto_find_test_dependencies()
endif()

# ========================================
# 8. 导出信息（供其他包使用）
# ========================================
ament_export_include_directories(include)
ament_export_dependencies(
  rclcpp
  behaviortree_cpp_v3
  rm_decision_interfaces
)

ament_package()  # ⚠️ 必须在文件末尾
```

---

### 6.3 常见编译错误诊断

#### ❌ **错误 1：`fatal error: xxx.hpp: No such file or directory`**

**原因**：编译器找不到头文件。

**解决步骤**：
```bash
# 1. 检查文件是否存在
ls include/rm_behavior_tree/plugins/condition/xxx.hpp

# 2. 检查 CMakeLists.txt 中的 include_directories
include_directories(include)  # ✅ 正确
include_directories(include/)  # ⚠️ 也可以（带不带 / 都行）

# 3. 检查 #include 路径
#include "rm_behavior_tree/plugins/condition/xxx.hpp"  # ✅ 相对于 include/
```

---

#### ❌ **错误 2：`undefined reference to 'rm_behavior_tree::IsControlZoneDetected::tick()'`**

**原因**：链接器找不到函数实现（.cpp 文件未编译）。

**解决步骤**：
```cmake
# 检查 add_executable 中是否包含了对应的 .cpp 文件
add_executable(rm_behavior_tree_node
  src/rm_behavior_tree.cpp
  plugins/condition/is_control_zone_detected.cpp  # ⚠️ 必须添加！
)
```

**调试技巧**：
```bash
# 查看哪些符号被编译进了可执行文件
nm -C build/rm_behavior_tree/rm_behavior_tree_node | grep IsControlZoneDetected

# 如果没有输出，说明该类根本没被编译
```

---

#### ❌ **错误 3：`undefined reference to 'typeinfo for rm_decision_interfaces::msg::RMUL'`**

**原因**：使用了自定义消息，但没有链接对应的库。

**解决方案**：
```cmake
# 确保在三个地方都添加了依赖
find_package(rm_decision_interfaces REQUIRED)  # 1. 查找包
ament_target_dependencies(rm_behavior_tree_node
  "rm_decision_interfaces"  # 2. 链接库
)
ament_export_dependencies(rm_decision_interfaces)  # 3. 导出依赖
```

---

### 6.4 优化编译速度的技巧

#### 🚀 **技巧 1：使用 ccache**
```bash
# 安装 ccache
sudo apt install ccache

# 在 CMakeLists.txt 中启用
set(CMAKE_CXX_COMPILER_LAUNCHER ccache)

# 效果：第二次编译速度提升 5-10 倍
```

#### 🚀 **技巧 2：并行编译**
```bash
# 使用所有 CPU 核心
colcon build --parallel-workers $(nproc)

# 或者指定核心数
colcon build --parallel-workers 8
```

#### 🚀 **技巧 3：增量编译**
```bash
# 只编译修改过的包
colcon build --packages-select rm_behavior_tree

# 只编译依赖了某个包的包
colcon build --packages-up-to rm_behavior_tree
```

---

### 6.5 CMake 调试工具

#### 🔍 **生成编译数据库（用于 IDE 代码补全）**
```bash
colcon build --cmake-args -DCMAKE_EXPORT_COMPILE_COMMANDS=ON

# 生成的文件位于
ls build/rm_behavior_tree/compile_commands.json
```

#### 🔍 **查看详细编译过程**
```bash
# 显示完整的编译命令
colcon build --event-handlers console_direct+

# 或者使用 make 的详细模式
colcon build --cmake-args -DCMAKE_VERBOSE_MAKEFILE=ON
```

#### 🔍 **检查依赖关系**
```bash
# 查看包依赖
colcon list --packages-above rm_behavior_tree

# 查看库依赖
ldd install/rm_behavior_tree/lib/rm_behavior_tree/rm_behavior_tree_node
```

## 7. 🔌 节点注册与 XML 配置 - 连接一切的桥梁

### 7.1 节点注册的两种方式

写好了 C++ 类，必须告诉 Factory 它叫什么名字，行为树才能构建出来。

#### 🔹 方式 A：手动注册（推荐用于主程序）

在 `main.cpp` 中显式注册。优点是编译期检查，简单直观。

```cpp
// src/rm_behavior_tree.cpp
#include "rm_behavior_tree/plugins/condition/is_control_zone_detected.hpp"
#include "rm_behavior_tree/plugins/action/send_goal.hpp"

// ...

int main(int argc, char ** argv)
{
    rclcpp::init(argc, argv);
    auto node = rclcpp::Node::make_shared("rm_behavior_tree");
    
    BT::BehaviorTreeFactory factory;

    // 1. 注册普通节点（不需要 ROS 节点句柄）
    factory.registerNodeType<rm_behavior_tree::IsControlZoneDetected>("IsControlZoneDetected");
    
    // 2. 注册带参数的节点（如需要 ROS 节点句柄）
    // 使用 lambda 表达式进行依赖注入
    factory.registerBuilder<rm_behavior_tree::SendGoalToNav2>(
        "SendGoal",
        [node](const std::string& name, const BT::NodeConfiguration& config) {
            return std::make_unique<rm_behavior_tree::SendGoalToNav2>(name, config, node);
        });

    // ...
}
```

#### 🔹 方式 B：插件式加载（高级）

将节点编译为 `.so` 动态库，在运行时加载。优点是无需重新编译主程序即可添加新节点。

```cpp
// 在节点 cpp 文件末尾添加宏
BT_REGISTER_NODES(factory)
{
    factory.registerNodeType<IsControlZoneDetected>("IsControlZoneDetected");
}
```

```cpp
// 在 main.cpp 中加载
factory.registerFromPlugin("./librm_behavior_tree_plugins.so");
```

---

### 7.2 XML 配置全解

#### 📄 `TreeNodesModel` 的作用

XML 文件底部的 `<TreeNodesModel>` 部分非常重要，它用于：
1. **静态检查**：Groot2 图形化工具用它来验证连接。
2. **运行时检查**：Factory 加载时会比对 C++ 代码中的 `providedPorts()`。

```xml
<root main_tree_to_execute="MainTree">
    <BehaviorTree ID="MainTree">
        <Sequence>
            <IsControlZoneDetected game_info="{game_state}"/>
            <SendGoal goal_pose="{target_pose}" timeout="10.0"/>
        </Sequence>
    </BehaviorTree>

    <!-- ⚠️ 必须与 C++代码中的 providedPorts() 严格一致 -->
    <TreeNodesModel>
        <Condition ID="IsControlZoneDetected">
            <input_port name="game_info" type="rm_decision_interfaces::msg::RMUL"/>
        </Condition>
        <Action ID="SendGoal">
            <input_port name="goal_pose"/>
            <input_port name="timeout" default="30.0"/>  <!-- 支持默认值 -->
            <output_port name="failure_reason"/>
        </Action>
    </TreeNodesModel>
</root>
```

#### 🔍 常见 XML 错误

| 错误现象 | 可能原因 | 解决方法 |
|---------|---------|----------|
| `Unknown Node Type` | 忘记在 main.cpp 注册节点 | 添加 `factory.registerNodeType` |
| `Attribute 'xxx' not found` | XML 参数名与 `providedPorts` 不一致 | 检查拼写，确保大小写匹配 |
| `Error parsing XML` | XML 语法错误（如标签未闭合） | 使用 `xmllint` 检查 |

**验证 XML 的神器**：
```bash
sudo apt install libxml2-utils
xmllint --noout config/3v3_new.xml
```

---

## 8. 🚀 最佳实践与高级技巧总结

### 8.1 调试清单 ✅

当你遇到问题时，按此清单排查：
1.  **编译期**：
    *   CMakeLists.txt 有没有添加 `.cpp` 文件？
    *   `includes` 目录对不对？
    *   依赖库链接了吗（`ament_target_dependencies`）？
2.  **配置期**：
    *   main.cpp 里 `registerNodeType` 字符串是否和 XML 里的标签名一致？
    *   XML 里的 `TreeNodesModel` 是否更新了？
3.  **运行期**：
    *   Blackboard 键名是否正确（`{key}` vs `"literal"`）？
    *   Port 类型是否匹配（C++ 类型检查）？
    *   是否有节点抛出了异常而没捕获？

### 8.2 ROS 2 集成建议

*   **尽量使用异步 Action**：涉及网络通信（Service/Action/Pub-Sub）的操作，耗时都不确定，尽量用 `StatefulActionNode`。
*   **不要在 tick() 中阻塞**：千万不要写 `sleep()` 或者长时间的 `while` 循环。tick() 必须在毫秒级返回，否则整个行为树会卡死，影响机器人的响应速度。
*   **使用命名空间**：封装你的自定义节点，防止与其他库冲突。

### 8.3 性能优化

*   **减少 Blackboard 读写**：如果数据不需要共享，就作为成员变量存储。
*   **控制 tick 频率**：使用 `RateController` 装饰器，或者在主循环中控制 `tree.tickOnce()` 的频率（通常 10Hz-50Hz 足够）。

---

## 🏁 结语

恭喜！你现在已经掌握了开发 BehaviorTree.CPP 自定义节点的全套技能。
从简单的条件判断，到复杂的异步导航任务，再到灵活的控制流，这套框架能帮你构建出强大且可维护的机器人决策系统。

**下一步做什么？**
*   尝试把现有的状态机代码重构为行为树节点。
*   使用 Groot2 可视化你的行为树运行状态。

*Happy Coding! 🤖*

---

## 9. 🔥 ROS 2 专用节点集成模式

### 9.1 使用 `behaviortree_ros2` 的预置节点

`behaviortree_ros2` 提供了开箱即用的 ROS 2 集成节点，无需自己实现。

#### 📡 **RosTopicSubNode - 订阅话题**

自动从 ROS 话题读取数据并更新到 Blackboard。

```cpp
// 在 main.cpp 中注册
BT::RosNodeParams params;
params.nh = node;
params.default_port_value = "game_state";  // 默认黑板键

factory.registerNodeType<BT::RosTopicSubNode<rm_decision_interfaces::msg::RMUL>>(
    "SubscribeGameState", params);
```

**XML 使用**：
```xml
<!-- 自动订阅 /referee/game_state 话题，并写入黑板 {game_info} -->
<SubscribeGameState topic="/referee/game_state" 
                    output_key="{game_info}" 
                    timeout="1.0"/>
```

#### 📤 **RosTopicPubNode - 发布话题**

从 Blackboard 读取数据并发布到 ROS 话题。

```cpp
factory.registerNodeType<BT::RosTopicPubNode<geometry_msgs::msg::Twist>>(
    "PublishVelocity", params);
```

**XML 使用**：
```xml
<PublishVelocity topic="/cmd_vel" 
                  message="{velocity_cmd}"/>
```

#### 🔧 **RosServiceNode - 调用服务**

```cpp
factory.registerNodeType<BT::RosServiceNode<std_srvs::srv::Trigger>>(
    "TriggerService", params);
```

---

### 9.2 实战案例：订阅传感器数据的完整流程

**场景**：订阅激光雷达数据，检查前方是否有障碍物。

```cpp
// 步骤 1: 定义 Condition 节点
class IsObstacleAhead : public BT::ConditionNode
{
public:
    IsObstacleAhead(const std::string& name, const BT::NodeConfiguration& config)
        : BT::ConditionNode(name, config) {}

    static BT::PortsList providedPorts() {
        return {
            BT::InputPort<sensor_msgs::msg::LaserScan>("scan_data"),
            BT::InputPort<double>("min_distance", 0.5, "安全距离（米）")
        };
    }

    BT::NodeStatus tick() override {
        sensor_msgs::msg::LaserScan scan;
        if (!getInput("scan_data", scan)) {
            return BT::NodeStatus::FAILURE;
        }

        double min_dist;
        getInput("min_distance", min_dist);

        // 检查前方 45 度范围内的最近障碍物
        size_t start_idx = scan.ranges.size() / 2 - 10;
        size_t end_idx = scan.ranges.size() / 2 + 10;

        for (size_t i = start_idx; i < end_idx; ++i) {
            if (scan.ranges[i] < min_dist) {
                return BT::NodeStatus::SUCCESS;  // 检测到障碍物
            }
        }
        return BT::NodeStatus::FAILURE;
    }
};
```

**XML 配置**：
```xml
<BehaviorTree ID="ObstacleAvoidance">
    <Sequence>
        <!-- 1. 订阅激光雷达数据 -->
        <SubscribeLaserScan topic="/scan" 
                            output_key="{laser_scan}" 
                            timeout="0.1"/>
        
        <!-- 2. 检查障碍物 -->
        <IsObstacleAhead scan_data="{laser_scan}" 
                         min_distance="0.8"/>
        
        <!-- 3. 执行避障动作 -->
        <PublishVelocity topic="/cmd_vel" 
                         message="{stop_cmd}"/>
    </Sequence>
</BehaviorTree>
```

---

## 10. 🐛 错误处理与异常安全

### 10.1 异常处理的黄金法则

**原则**：行为树节点不应该让异常逃逸到 Factory 层。

#### ❌ **错误示例**
```cpp
BT::NodeStatus tick() override {
    auto data = getInput<Data>("input");  // 可能抛出异常
    return process(data);  // 未捕获异常
}
```

#### ✅ **正确示例**
```cpp
BT::NodeStatus tick() override {
    try {
        Data data;
        if (!getInput("input", data)) {
            throw BT::RuntimeError("Missing required input 'input'");
        }
        
        return process(data);
        
    } catch (const BT::RuntimeError& e) {
        RCLCPP_ERROR(logger_, "RuntimeError in [%s]: %s", name().c_str(), e.what());
        return BT::NodeStatus::FAILURE;
        
    } catch (const std::exception& e) {
        RCLCPP_FATAL(logger_, "Unexpected exception in [%s]: %s", name().c_str(), e.what());
        return BT::NodeStatus::FAILURE;
    }
}
```

---

### 10.2 超时与资源管理

#### 🕒 **使用 RAII 管理资源**

```cpp
class SafeAction : public BT::StatefulActionNode
{
private:
    std::unique_ptr<ResourceHandle> resource_;

public:
    BT::NodeStatus onStart() override {
        // RAII: 资源自动管理
        resource_ = std::make_unique<ResourceHandle>();
        return BT::NodeStatus::RUNNING;
    }

    void onHalted() override {
        // resource_ 会自动析构，即使发生异常
        resource_.reset();  // 显式释放（可选）
    }
};
```

---

## 11. 🖥️ Groot2 可视化与调试

### 11.1 导出实时日志

在 C++ 代码中启用日志记录器：

```cpp
// main.cpp
int main(int argc, char** argv)
{
    // ...
    
    // 创建日志记录器（发布到 ROS 话题）
    BT::PublisherZMQ publisher_zmq(tree);
    
    while (rclcpp::ok()) {
        tree.tickOnce();
        publisher_zmq.flush();  // 发送状态到 Groot2
        rclcpp::spin_some(node);
        rate.sleep();
    }
}
```

### 11.2 在 Groot2 中连接

1. 打开 Groot2：`ros2 run groot2 Groot2`
2. 点击 **Monitor** 标签页
3. 连接到 `tcp://localhost:1666`
4. 实时查看节点状态变化

---

## 12. 📋 快速参考卡片

### 12.1 节点类型速查表

| 节点类型 | 基类 | 必须重写 | 可返回 RUNNING | 典型用途 |
|---------|------|---------|---------------|---------|
| **Condition** | `ConditionNode` | `tick()` | ❌ | 布尔检查 |
| **Sync Action** | `SyncActionNode` | `tick()` | ❌ | 瞬时操作 |
| **Stateful Action** | `StatefulActionNode` | `onStart()`, `onRunning()`, `onHalted()` | ✅ | 异步任务 |
| **Control** | `ControlNode` | `tick()`, `halt()` | ✅ | 控制流 |
| **Decorator** | `DecoratorNode` | `tick()` | ✅ | 修改行为 |

---

### 12.2 Port 类型速查

```cpp
// 输入端口（只读）
BT::InputPort<int>("count")
BT::InputPort<std::string>("name", "default_value", "描述")

// 输出端口（只写）
BT::OutputPort<bool>("result")

// 双向端口
BT::BidirectionalPort<double>("value")
```

---

### 12.3 常用调试命令

```bash
# 检查 XML 语法
xmllint --noout config/tree.xml

# 查看编译符号
nm -C build/package/executable | grep ClassName

# 检查库依赖
ldd install/package/lib/package/executable

# 运行时日志级别
ros2 run package node --ros-args --log-level debug

# 查看行为树节点注册情况（在代码中添加）
factory.logRegisteredNodes();
```

---

### 12.4 故障排查流程图

```
编译失败？
├─ 语法错误 → 检查 C++ 语法
├─ 找不到头文件 → 检查 include_directories
└─ undefined reference → 检查 .cpp 是否添加到 CMakeLists.txt

运行时崩溃？
├─ Segmentation Fault → 检查空指针、数组越界
├─ 未捕获异常 → 添加 try-catch
└─ ROS 通信失败 → 检查话题名称、消息类型

行为树不按预期执行？
├─ 节点未注册 → 检查 factory.registerNodeType
├─ XML 解析失败 → 使用 xmllint 验证
├─ Port 类型不匹配 → 检查 C++ 和 XML 的类型定义
└─ Blackboard 键名错误 → 检查 {key} 的拼写
```

---

## 13. 💡 生产环境最佳实践

### 13.1 日志分级策略

```cpp
// 根据严重性选择日志级别
RCLCPP_DEBUG(logger_, "Minor detail: %d", value);        // 仅调试时
RCLCPP_INFO(logger_, "Normal operation: started");       // 关键状态变化
RCLCPP_WARN(logger_, "Unexpected but recoverable: %s", msg);  // 异常但能继续
RCLCPP_ERROR(logger_, "Failed: %s", reason);             // 功能失败
RCLCPP_FATAL(logger_, "Critical error, shutting down");  // 系统级错误
```

### 13.2 性能监控

```cpp
// 在关键节点添加性能计时
class MonitoredAction : public BT::StatefulActionNode
{
    BT::NodeStatus onStart() override {
        start_time_ = std::chrono::high_resolution_clock::now();
        // ...
    }

    BT::NodeStatus onRunning() override {
        auto elapsed = std::chrono::high_resolution_clock::now() - start_time_;
        auto ms = std::chrono::duration_cast<std::chrono::milliseconds>(elapsed).count();
        
        if (ms > 100) {  // 超过 100ms 警告
            RCLCPP_WARN(logger_, "[%s] Taking too long: %ld ms", name().c_str(), ms);
        }
        // ...
    }
};
```

### 13.3 单元测试示例

```cpp
// test/test_is_control_zone_detected.cpp
#include <gtest/gtest.h>
#include "rm_behavior_tree/plugins/condition/is_control_zone_detected.hpp"

TEST(IsControlZoneDetectedTest, ReturnsSuccessWhenCaptured)
{
    BT::BehaviorTreeFactory factory;
    factory.registerNodeType<rm_behavior_tree::IsControlZoneDetected>("TestNode");
    
    auto tree = factory.createTreeFromText(R"(
        <root>
            <BehaviorTree>
                <TestNode game_info="{game}"/>
            </BehaviorTree>
        </root>
    )");
    
    // 设置测试数据
    rm_decision_interfaces::msg::RMUL game_data;
    game_data.control_zone_captured = true;
    tree.rootBlackboard()->set("game", game_data);
    
    // 执行并验证
    auto status = tree.tickOnce();
    EXPECT_EQ(status, BT::NodeStatus::SUCCESS);
}
```

---

## 🎓 进阶学习资源

### 官方文档
- [BehaviorTree.CPP 官方文档](https://www.behaviortree.dev/)
- [Groot2 可视化工具](https://github.com/BehaviorTree/Groot2)
- [ROS 2 BehaviorTree 集成](https://github.com/BehaviorTree/BehaviorTree.ROS2)

### 推荐阅读
- 《Game AI Pro》- 第 3 章：Behavior Trees
- 《Artificial Intelligence for Games》- Millington & Funge

### 实战项目参考
- Nav2 导航栈的行为树实现
- MoveIt2 机械臂规划中的决策树